[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BartekDaniluk/Big_Data/blob/main/Python_type_validation.ipynb)


# Walidacja typów danych w pythonie

In [21]:
string: str = 'Jakiś string'
integer: int = 1234

In [22]:
def create_car(brand: str, model: str = None, displacement: float | None = None) -> dict[str, str | float | None]:
  return {
      'brand': brand,
      'model': model,
      'displacement': displacement
  }

In [23]:
create_car('Ford', 'Mustang boss 429', 7.0)

{'brand': 'Ford', 'model': 'Mustang boss 429', 'displacement': 7.0}

In [24]:
type Car = dict[str, str | float | None]

def create_car(brand: str, model: str = None, displacement: float | None = None) -> Car:
  return {
      'brand': brand,
      'model': model,
      'displacement': displacement
  }

In [25]:
create_car('Nissan', 'Skyline R32', 3.0)

{'brand': 'Nissan', 'model': 'Skyline R32', 'displacement': 3.0}

In [26]:
from typing import NewType

four_speed_gear_ratios = NewType('four_speed_gear_ratios', tuple[float, float, float, float, float])
type Car = dict[str, str | float | four_speed_gear_ratios | None]

def create_car(brand: str, model: str, gear_ratios: four_speed_gear_ratios, displacement: float | None = None) -> Car:
  return {
      'brand': brand,
      'model': model,
      'gear_ratios': gear_ratios,
      'displacement': displacement
  }

create_car('Ford', 'Mustang boss 429', four_speed_gear_ratios((2.32, 1.69, 1.29, 1.0 , 2.32)), 7.0)

{'brand': 'Ford',
 'model': 'Mustang boss 429',
 'gear_ratios': (2.32, 1.69, 1.29, 1.0, 2.32),
 'displacement': 7.0}

In [27]:
from typing import NewType, TypedDict

four_speed_gear_ratios = NewType('four_speed_gear_ratios', tuple[float, float, float, float, float])

class Car(TypedDict):
  brand: str
  model: str
  gear_ratio: four_speed_gear_ratios
  displacement: float | None

def create_car(brand: str, model: str, gear_ratios: four_speed_gear_ratios, displacement: float | None = None) -> Car:
  return {
      'brand': brand,
      'model': model,
      'gear_ratios': gear_ratios,
      'displacement': displacement
  }
create_car('Ford', 'Mustang boss 429', four_speed_gear_ratios((2.32, 1.69, 1.29, 1.0 , 2.32)), 7.0)

{'brand': 'Ford',
 'model': 'Mustang boss 429',
 'gear_ratios': (2.32, 1.69, 1.29, 1.0, 2.32),
 'displacement': 7.0}

### Dataclasses – "Standardowe" podejście do kontenerów danych

Wprowadzone w Pythonie 3.7, `dataclasses` automatyzują tworzenie metod takich jak `__init__`, `__repr__` czy `__eq__`. W przeciwieństwie do `TypedDict`, tworzą one prawdziwą klasę (obiekt), a nie tylko podpowiedź dla słownika.

In [ ]:
from dataclasses import dataclass

@dataclass
class CarData:
    brand: str
    model: str
    gear_ratios: four_speed_gear_ratios
    displacement: float | None = None

car_obj = CarData('Ford', 'Mustang boss 429', four_speed_gear_ratios((2.32, 1.69, 1.29, 1.0, 2.32)), 7.0)
print(car_obj)

### Nominalne vs Strukturalne Typowanie (Duck Typing)

Python tradycyjnie opiera się na "Duck Typing" (jeśli kwacze jak kaczka, to jest kaczką). 
- **Typowanie Nominalne** (np. `class Car(Vehicle)`): Obiekt musi jawnie dziedziczyć po danej klasie.
- **Typowanie Strukturalne** (`typing.Protocol`): Obiekt musi po prostu posiadać wymagane metody i atrybuty. To doskonałe narzędzie do tworzenia elastycznych interfejsów bez wymuszania hierarchii klas.

In [28]:
from typing import Protocol

class Movable(Protocol):
    def move(self) -> None: ...

class Robot:
    def move(self) -> None:
        print("Robot jedzie...")

class Bird:
    def move(self) -> None:
        print("Ptak leci...")

def start_movement(obj: Movable):
    obj.move()

start_movement(Robot())
start_movement(Bird())

Robot jedzie...
Ptak leci...


# Pydantic

In [29]:
import pydantic

pydantic.__version__

'2.7.1'

W przeciwieństwie do normalengo pythonowskiego typehintingu, obiekty dziedziczące po pydanticowym BaseModel, są sprawdzane podczas runtime, w kontkeście zgodności danych. Wszystko musie się zgadzać.

In [30]:
from datetime import datetime
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    uid: int
    username: str
    date_of_birth: datetime | None = None
    is_active: bool = True

    first_name: str | None = None
    surname: str | None = None

In [31]:
u1 = User(
    uid = 112,
    username = "Username_1",
    first_name = "Name1",
    surname = "Surname1"
)

Jeśli coś się nie zgadza, od razu rzucany jest ValidationError, który można ładnie przechwycić. Z ciekawostek pydantic ma domyslnie włączone typeconversion. Czyli jeśli w int wrzucimu '123', nit będzie problemu, zostanie to zamieniane na int. Poód do tego jest taki że pozyskiwanie numerycznych wartości, w postaci stringów, jest dośc popularne. Czy to z jsona, parametrów URL itd...

In [32]:
try:
    u12 = User(
        uid = '1123',
        username = 123123,
        first_name = "Name1",
        surname = "Surname1"
    )
except ValidationError as e:
    print(e)

1 validation error for User
username
  Input should be a valid string [type=string_type, input_value=123123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.7/v/string_type


In [33]:
print(u1)

uid=112 username='Username_1' date_of_birth=None is_active=True first_name='Name1' surname='Surname1'


Wspiera również natywną serializację.

In [34]:
u1.model_dump_json(indent=2)

'{\n  "uid": 112,\n  "username": "Username_1",\n  "date_of_birth": null,\n  "is_active": true,\n  "first_name": "Name1",\n  "surname": "Surname1"\n}'

### Dlaczego `default_factory`? Problem Mutable Defaults

W Pythonie domyślne argumenty funkcji (i pól klas) są ewaluowane **tylko raz** – w momencie definicji. Jeśli użyjemy obiektu mutowalnego (np. `list`, `dict`), ten sam obiekt będzie dzielony między wszystkie instancje klasy.

**Przykład błędu:**
```python
def add_to_list(val, my_list=[]): # To samo [] dla każdego wywołania!
    my_list.append(val)
    return my_list
```

`default_factory` rozwiązuje ten problem, przyjmując funkcję, która zostanie wywołana za każdym razem, gdy potrzebna jest nowa domyślna wartość.

### Czym jest `functools.partial`?

`partial` to funkcja wyższego rzędu, która pozwala na "zamrożenie" części argumentów innej funkcji, tworząc nowy obiekt wywoływalny o uproszczonej sygnaturze. Jest to niezwykle przydatne właśnie w `default_factory`, gdy chcemy przekazać parametry do funkcji tworzącej (np. strefę czasową do `datetime.now`).

In [35]:
from pydantic import Field
import pytz
from functools import partial
from typing import Literal

class SupportTicket(BaseModel):
    uid: int
    sender_id: int
    content: str
    priority: Literal['low', 'medium', 'high'] = 'low'
    is_resolved: bool = False

    tages: list[str] = Field(default_factory=list)
    created_at: datetime = Field(default_factory=partial(datetime.now, pytz.timezone('Europe/Berlin')))

    status: Literal['unresolved', 'being resolved', 'resolved'] = 'unresolved'

In [36]:
 st1 = SupportTicket(
    uid = 221,
    sender_id = u1.uid,
    content = "Test content",
    tags = ['tag1', 'tag2'],
    status = 'being resolved'
 )

In [37]:
st1.model_dump_json(indent=2)

'{\n  "uid": 221,\n  "sender_id": 112,\n  "content": "Test content",\n  "priority": "low",\n  "is_resolved": false,\n  "tages": [],\n  "created_at": "2026-04-13T01:12:18.263407+02:00",\n  "status": "being resolved"\n}'

Jeśłi chodzi o Field, to ma wiele więcej możłiwości. Możdna dodawać wartości min max, długośc, wzory (regexy) itd... Zeby to zaimplementować, wykorzystujemy type Annotated z pythonowskiego typing

In [38]:
from typing import Annotated
from pydantic import Field

class User(BaseModel):
    uid: Annotated[int, Field(gt=0)]
    email: Annotated[str, Field(pattern=r"^[^@]+@[^@]+\\.[^@]+$")]
    username: Annotated[str, Field(min_length=6, max_length=30)]
    date_of_birth: datetime | None = None
    is_active: bool = True

    first_name: str | None = None
    surname: str | None = None

Czyli można sprawdzać maile własnoręcze, ale po co się męczyć? Pydantic może to zrobić za nas. Np, z Emailami, Hasłem, URL.

In [52]:
from pydantic import HttpUrl, SecretStr, EmailStr

class User(BaseModel):
    uid: Annotated[int, Field(gt=0)]
    email: EmailStr
    password: SecretStr
    users_url: HttpUrl | None
    username: Annotated[str, Field(min_length=6, max_length=30)]
    date_of_birth: datetime | None = None
    is_active: bool = True

    first_name: str | None = None
    surname: str | None = None

In [54]:
try:
    u = User(
        uid = 111,
        email = 'mail123@gmail.com',
        password = 'Haslo!23',
        users_url = 'https://www.myfitnesspal.com/food/diary/mail123',
        username = 'MisioPysio69',
    )
except ValidationError as e:
    print(e)

In [55]:
print(u)

uid=111 email='mail123@gmail.com' password=SecretStr('**********') users_url=Url('https://www.myfitnesspal.com/food/diary/mail123') username='MisioPysio69' date_of_birth=None is_active=True first_name=None surname=None


In [56]:
print(u.password.get_secret_value())

Haslo!23


# Architektura: Cykl życia danych i LangGraph

Na podstawie analizy "Python Typing vs. Pydantic Comparison", kluczowym aspektem projektowania systemów Big Data i AI jest wybór odpowiedniego narzędzia do odpowiedniego zadania. Nie zawsze Pydantic jest najlepszym wyborem.

### Data Lifecycle Heuristic Architecture (Heurystyka Cyklu Życia Danych)

| Faza Cyklu | Charakterystyka | Rekomendacja | Uzasadnienie |
| :--- | :--- | :--- | :--- |
| **Ingress (Wejście)** | Dane z zewnątrz (API, LLM, User), niepewne, surowe. | **Pydantic** | Rygorystyczna walidacja na brzegach systemu. |
| **Compute (Obliczenia)** | Wewnętrzna logika, wysoka częstotliwość, zaufane dane. | **TypedDict / Dataclasses** | Minimalny overhead, maksymalna szybkość procesora. |
| **Persist (Zapis)** | Serializacja do bazy, zachowanie stanu. | **Dataclasses** | Czystość logiczna i brak narzutu walidacji przy odczycie z zaufanego źródła. |
| **Serve (Wyjście)** | Odpowiedzi HTTP, kontrakty API. | **Pydantic** | Gwarancja zgodności z dokumentacją (OpenAPI). |

### Przypadek LangGraph: Balancing Validation and Velocity

W frameworkach agentowych takich jak **LangGraph**, stan (State) przechodzi przez dziesiątki lub setki węzłów. 
- Użycie `Pydantic` jako stanu grafu powoduje walidację całego obiektu przy każdym przejściu między węzłami (overload).
- Dlatego jako **Internal State** zaleca się `TypedDict` – daje nam to wsparcie IDE i autouzupełnianie, ale z zerowym kosztem w runtime.
- Dopiero dane wyjściowe z LLM (Structured Output) powinny przechodzić przez `Pydantic` przed umieszczeniem ich w zaufanym stanie grafu.

In [40]:
from typing import Annotated, TypedDict
import operator

# 1. WEWNĘTRZNY STAN GRAFU (Wydajność: TypedDict)
class AgentState(TypedDict):
    # Annotated tutaj służy LangGraphowi do określenia jak łączyć (reducer) wiadomości
    messages: Annotated[list[str], operator.add]
    confidence_score: float
    iteration: int

# 2. GRANICA SYSTEMU / WYJŚCIE LLM (Bezpieczeństwo: Pydantic)
class ExtractionResult(BaseModel):
    key_insights: list[str] = Field(min_length=1, description="Główne wnioski z analizy")
    confidence_score: float = Field(ge=0.0, le=1.0)

    @model_validator(mode='after')
    def verify_confidence_pairing(self) -> 'ExtractionResult':
        if len(self.key_insights) < 3 and self.confidence_score > 0.9:
            raise ValueError("Wysoka pewność wymaga przynajmniej 3 wniosków wspierających.")
        return self

print("Zaprojektowano hybrydowy system walidacji.")

NameError: name 'model_validator' is not defined

In [ ]:
u2 = User(
    uid = 11,
    username='aaoliudhfalksjhdfalksjdhflaksjdhfalskjdhfalskjdhfalskjdhfaslkjdfhalskdjfh')

ValidationError: 1 validation error for User
username
  String should have at most 30 characters [type=string_too_long, input_value='aaoliudhfalksjhdfalksjdh...lskjdhfaslkjdfhalskdjfh', input_type=str]
    For further information visit https://errors.pydantic.dev/2.7/v/string_too_long